# ShopUNow — Create Department Vector Databases

This notebook creates **separate persistent ChromaDB vector databases** for each of the 7 ShopUNow departments.  
Each department's QA pairs are embedded and stored in their own collection under `vectorstores/<department>/`.  
These databases will be loaded by the Agentic RAG system for department-specific retrieval.

In [ ]:
# ============================================================================
# Imports
# ============================================================================

import json
import os
import sys
import shutil
from pathlib import Path

import chromadb
from langchain_chroma import Chroma

# Add project root so helpers module is importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from helpers.utils import get_embeddings

In [ ]:
# ============================================================================
# Configuration
# ============================================================================

DATA_DIR = Path("data")
VECTORSTORE_DIR = Path("vectorstores")

# Department name → JSON filename (without .json)
DEPARTMENTS = {
    "hr":                 "hr_data",
    "it_support":         "it_support_data",
    "facilities_admin":   "facilities_admin_data",
    "customer_service":   "customer_service_data",
    "product_sales":      "product_sales_data",
    "billing_payments":   "billing_payments_data",
    "shipping_delivery":  "shipping_delivery_data",
}

# Initialize embedding function (auto-selects by platform)
embeddings = get_embeddings()

In [ ]:
# ============================================================================
# Load All JSON Data Files
# ============================================================================

department_data: dict[str, list[dict]] = {}

for dept, filename in DEPARTMENTS.items():
    filepath = DATA_DIR / f"{filename}.json"
    with open(filepath) as f:
        records = json.load(f)
    department_data[dept] = records
    print(f"{dept:<22} — {len(records):>3} QA pairs  ({filepath})")

total = sum(len(v) for v in department_data.values())
print(f"{'TOTAL':<22} — {total:>3} QA pairs across {len(department_data)} departments")

In [ ]:
# ============================================================================
# Create ChromaDB Vector Database for a Single Department
# ============================================================================

def collection_name_for(dept_name: str) -> str:
    """Return a ChromaDB-safe collection name (min 3 chars required)."""
    return f"dept_{dept_name}"

def create_department_vectorstore(
    dept_name: str,
    records: list[dict],
    embeddings,
    base_dir: Path = VECTORSTORE_DIR,
) -> Chroma:
    """Create a persistent ChromaDB vectorstore for one department."""
    persist_dir = str(base_dir / dept_name)

    # Remove existing directory to avoid stale data
    if os.path.exists(persist_dir):
        shutil.rmtree(persist_dir)

    documents = [r["doc"] for r in records]
    metadatas = [{"category": r["category"]} for r in records]

    vectorstore = Chroma.from_texts(
        texts=documents,
        metadatas=metadatas,
        embedding=embeddings,
        persist_directory=persist_dir,
        collection_name=collection_name_for(dept_name),
    )

    return vectorstore

In [ ]:
# ============================================================================
# Create All 7 Vector Databases
# ============================================================================

VECTORSTORE_DIR.mkdir(exist_ok=True)
vectorstores: dict[str, Chroma] = {}

for dept, records in department_data.items():
    print(f"Creating vectorstore for {dept} ...", end=" ")
    vs = create_department_vectorstore(dept, records, embeddings)
    vectorstores[dept] = vs
    print(f"done  ({vs._collection.count()} vectors)")

print(f"\nAll {len(vectorstores)} vectorstores created under {VECTORSTORE_DIR}/")

In [ ]:
# ============================================================================
# Verification — Reload & Query Each Database
# ============================================================================

SAMPLE_QUERIES = {
    "hr":                 "What is the leave policy?",
    "it_support":         "How do I reset my password?",
    "facilities_admin":   "How do I book a meeting room?",
    "customer_service":   "How do I handle a customer complaint?",
    "product_sales":      "What are the current product discounts?",
    "billing_payments":   "How do I process a refund?",
    "shipping_delivery":  "How do I track my shipment?",
}

print("Reloading vectorstores from disk and running sample queries ...\n")

for dept in DEPARTMENTS:
    persist_dir = str(VECTORSTORE_DIR / dept)
    vs = Chroma(
        persist_directory=persist_dir,
        embedding_function=embeddings,
        collection_name=collection_name_for(dept),
    )
    query = SAMPLE_QUERIES[dept]
    results = vs.similarity_search(query, k=1)

    print(f"[{dept}]  Query: \"{query}\"")
    print(f"  Top result: {results[0].page_content[:120]}...")
    print()

In [ ]:
# ============================================================================
# Summary Statistics
# ============================================================================

def dir_size_mb(path: str) -> float:
    total = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())
    return total / (1024 * 1024)

print(f"{'Department':<22} {'Docs':>5} {'Size (MB)':>10}  Path")
print("-" * 70)

for dept in DEPARTMENTS:
    persist_dir = str(VECTORSTORE_DIR / dept)
    vs = Chroma(
        persist_directory=persist_dir,
        embedding_function=embeddings,
        collection_name=collection_name_for(dept),
    )
    count = vs._collection.count()
    size = dir_size_mb(persist_dir)
    print(f"{dept:<22} {count:>5} {size:>10.2f}  {persist_dir}")

print("-" * 70)
total_size = dir_size_mb(str(VECTORSTORE_DIR))
print(f"{'TOTAL':<22} {total:>5} {total_size:>10.2f}  {VECTORSTORE_DIR}/")
print("\nAll vector databases are ready for the Agentic RAG system.")